<a href="https://colab.research.google.com/github/nterreri-cmu/p2-datacenters/blob/main/Python_Data_Centers_Nguyen_Terreri.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Data Centers and Energy Use: An Exploratory Data Analysis
by Vi Nguyen and Natali Terreri
Python Programming II, Fall 2026

In [1]:
# import Pandas
import pandas as pd

# import numpy
import numpy as np

#matplot
import matplotlib.pyplot as plt

#seaborn
import seaborn as sns

#api calls
import requests # to download
import duckdb # to query



## 1. Setup Data in Python

### Upload data as the following:


1.   data centers from Fracktracker (datacenters_df)
2.   electricity usage (sales) from EIA (electricity_df)
3.   generator data from EIA

      *   current generators (current_gen_df)
      *   proposed generators (proposed_gen_df)
      *   all generators (generators_df)



In [2]:
#upload data center data file

dc_url = 'https://raw.githubusercontent.com/nterreri-cmu/p2-datacenters/refs/heads/main/Data_Centers_Database_Copy%20-%20FracTracker%20Data%20Centers.csv'

datacenters_df = pd.read_csv(dc_url)


In [18]:
display(datacenters_df)

,facility_name,address,city,state,zip,county,lat,long,status,location_confidence,...,info_source_1,info_source_2,info_source_3,info_source_4,info_source_5,info_source_6,info_source_7,info_source_8,date_created,date_updated
0,Stak Energy Data Center,"Dalton Hwy, 26 miles south of Deadhorse",North Slope Borough,AK,99734.0,North Slope,69.900710,-148.81477,Proposed,Medium,...,https://www.datacenterdynamics.com/en/news/sta...,https://www.cbc.ca/news/canada/north/yukon-ala...,https://alaskapublic.org/news/economy/energy/2...,https://alaskabeacon.com/2026/07/06/data-cente...,https://aws.state.ak.us/OnlinePublicNotices/No...,NaN,NaN,NaN,05/20/2026,07/08/2026
1,Prudhoe Bay Data Center,Dalton Hwy,Prudhoe Bay,AK,99734.0,North Slope,70.184780,-148.44000,Operating,Medium,...,https://www.fn-digital.com/data-center,https://www.datacenterdynamics.com/en/news/sta...,NaN,NaN,NaN,NaN,NaN,NaN,05/20/2026,05/20/2026
2,Project Marvel,Red Ridge on Rock Mountain Lake Rd and Old Law...,Bessemer,AL,35022.0,Jefferson,33.342000,-87.03410,Proposed,High,...,https://www.datacenterdynamics.com/en/news/700...,https://www.youtube.com/watch?v=ICSrvQ7meow&ab...,https://www.al.com/news/2025/06/14-billion-pro...,https://www.wbrc.com/2025/09/14/bessemer-resid...,https://www.wbrc.com/2025/11/19/bessemer-city-...,https://www.datacenterdynamics.com/en/news/ala...,https://abc3340.com/news/abc-3340-news-iteam/b...,https://insideclimatenews.org/news/13112025/pr...,09/16/2025,07/08/2026
3,BHM01 Nebius Data Center,201 Milan Parkway,Birmingham,AL,35221.0,Jefferson,33.427460,-86.88143,Proposed,High,...,https://www.datacenterdynamics.com/en/news/neb...,https://www.bhamyouthfirst.org/news/city-birmi...,https://assets.nebius.com/assets/69962c26-79c9...,https://abc3340.com/news/local/birmingham-city...,NaN,NaN,NaN,NaN,02/20/2026,07/08/2026
4,DC Blox,433 6th St S,Birmingham,AL,35233.0,Jefferson,33.500510,-86.82100,Operating,High,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,08/25/2025,08/25/2025
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1696,Microsoft Azure Data Center,644 Logistics Dr,Cheyenne,WY,82009.0,Laramie,41.123720,-104.90100,Approved/Permitted/Under construction,High,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,08/11/2025,08/11/2025
1697,Microsoft Cheyenne Data Center,Near Wapiti Trail,Cheyenne,WY,82007.0,Laramie,41.064470,-104.77278,Proposed,Medium,...,https://www.datacenterdynamics.com/en/news/mic...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,04/15/2026,04/15/2026
1698,Project Tembo/ now Project Braid,"Switchgrass Industrial Park, west of I-85. 2 m...",Cheyenne,WY,82007.0,Laramie,41.031693,-104.79640,Suspended,High,...,https://capcity.news/community/county-communit...,https://www.washingtonpost.com/climate-environ...,https://www.crusoe.ai/resources/newsroom/cruso...,https://www.wyomingnews.com/news/local_news/co...,https://www.datacenterdynamics.com/en/news/cru...,https://www.datacenterdynamics.com/en/news/cru...,https://www.datacenterdynamics.com/en/news/goo...,https://www.datacenterdynamics.com/en/news/goo...,12/08/2025,09/10/2026
1699,Related Digital Cheyenne Data Center,"Cheyenne Business Parkway, Campstool Rd, north...",Cheyenne,WY,82007.0,Laramie,41.137573,-104.71874,Proposed,Medium,...,https://oilcity.news/statewide/2025/10/07/1-2-...,https://www.datacenterdynamics.com/en/news/rel...,https://cheyenneleads.org/wp-content/uploads/2...,NaN,NaN,NaN,NaN,NaN,12/08/2025,12/08/2025


In [4]:
#get api key for EIA data
from google.colab import userdata
eia_key = userdata.get('EIA_Key')

In [6]:
#upload electricity sale data
#API for Electricity Sales to Ultimate Customers (monthly)
#http://api.eia.gov/v2/electricity/retail-sales/data/?api_key=xxxxxx&facets

#url = f'https://api.eia.gov/v2/electricity/retail-sales/data/?api_key={eia_key}&frequency=monthly&data[0]=customers&data[1]=price&data[2]=revenue&data[3]=sales&start=2016-01&end=2026-07&sort[0][column]=period&sort[0][direction]=desc&offset=0&length=5000'

#API for Electricity Sales to Ultimate Customers (annual)

eia_url = f'https://api.eia.gov/v2/electricity/retail-sales/data/?api_key={eia_key}&frequency=annual&data[0]=customers&data[1]=price&data[2]=revenue&data[3]=sales&start=2016&end=2026&sort[0][column]=period&sort[0][direction]=desc&offset=0&length=5000'

response = requests.get(eia_url)
print(response.status_code)

200


In [7]:
#get electricity usage as json file
electricity = response.json()
print(type(electricity))
print(electricity)

<class 'dict'>
{'response': {'total': '3720', 'dateFormat': 'YYYY', 'frequency': 'annual', 'data': [{'period': '2025', 'stateid': 'AK', 'stateDescription': 'Alaska', 'sectorid': 'ALL', 'sectorName': 'all sectors', 'customers': '364851', 'price': '23.05', 'revenue': '1410.4809', 'sales': '6118.4264', 'customers-units': 'number of customers', 'price-units': 'cents per kilowatt-hour', 'revenue-units': 'million dollars', 'sales-units': 'million kilowatt hours'}, {'period': '2025', 'stateid': 'AK', 'stateDescription': 'Alaska', 'sectorid': 'COM', 'sectorName': 'commercial', 'customers': '58188', 'price': '22.32', 'revenue': '592.43925', 'sales': '2654.25995', 'customers-units': 'number of customers', 'price-units': 'cents per kilowatt-hour', 'revenue-units': 'million dollars', 'sales-units': 'million kilowatt hours'}, {'period': '2025', 'stateid': 'AK', 'stateDescription': 'Alaska', 'sectorid': 'IND', 'sectorName': 'industrial', 'customers': '1240', 'price': '20.03', 'revenue': '283.44752',

In [15]:
#flatten json data and convert to dataframe
electricity_df = pd.json_normalize(electricity['response']['data'])

display(electricity_df)

,period,stateid,stateDescription,sectorid,sectorName,customers,price,revenue,sales,customers-units,price-units,revenue-units,sales-units
0,2025,AK,Alaska,ALL,all sectors,364851,23.05,1410.4809,6118.4264,number of customers,cents per kilowatt-hour,million dollars,million kilowatt hours
1,2025,AK,Alaska,COM,commercial,58188,22.32,592.43925,2654.25995,number of customers,cents per kilowatt-hour,million dollars,million kilowatt hours
2,2025,AK,Alaska,IND,industrial,1240,20.03,283.44752,1415.16336,number of customers,cents per kilowatt-hour,million dollars,million kilowatt hours
3,2025,AK,Alaska,OTH,other,0,0,0,0,number of customers,cents per kilowatt-hour,million dollars,million kilowatt hours
4,2025,AK,Alaska,RES,residential,305423,26.09,534.59413,2049.00309,number of customers,cents per kilowatt-hour,million dollars,million kilowatt hours
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3715,2016,WY,Wyoming,COM,commercial,57434,9.4,353.6602,3762.315,number of customers,cents per kilowatt-hour,million dollars,million kilowatt hours
3716,2016,WY,Wyoming,IND,industrial,10654,6.92,695.16609,10041.482,number of customers,cents per kilowatt-hour,million dollars,million kilowatt hours
3717,2016,WY,Wyoming,OTH,other,0,0,0,0,number of customers,cents per kilowatt-hour,million dollars,million kilowatt hours
3718,2016,WY,Wyoming,RES,residential,269647,11.13,306.2976,2751.073,number of customers,cents per kilowatt-hour,million dollars,million kilowatt hours


In [19]:
#upload generator info to dataframes

#current generators
current = 'https://raw.githubusercontent.com/nterreri-cmu/p2-datacenters/refs/heads/main/EIA_Generator_Y2025_Current.csv'
current_gen_df = pd.read_csv(current)

#proposed generators
proposed = 'https://raw.githubusercontent.com/nterreri-cmu/p2-datacenters/refs/heads/main/EIA_Generator_Y2025_Proposed.csv'
proposed_gen_df = pd.read_csv(proposed)



/tmp/ipykernel_973/194418413.py:5: DtypeWarning: Columns (0,15,17,18,35,36,37,38,41,42,51,52,53) have mixed types. Specify dtype option on import or set low_memory=False.
  current_gen_df = pd.read_csv(current)


In [ ]:
#clean current_gen_df

#add status column = "Current"

#only keep necessary columns

#correct NaN or other types

In [ ]:
#clean planned_gen_df

#add status column = "Planned"

#only keep necessary columns

#correct NaN or other types

In [20]:
display(current_gen_df)

,Utility ID,Utility Name,Plant Code,Plant Name,State,County,Generator ID,Technology,Prime Mover,Unit Code,...,Planned Energy Source 1,Planned New Nameplate Capacity (MW),Planned Repower Month,Planned Repower Year,Other Planned Modifications?,Other Modifications Month,Other Modifications Year,Multiple Fuels?,Cofire Fuels?,Switch Between Oil and Natural Gas?
0,63560,"Sand Point Generating, LLC",1.0,Sand Point,AK,Aleutians East,1,Petroleum Liquids,IC,NaN,...,NaN,,,,NaN,,,N,NaN,NaN
1,63560,"Sand Point Generating, LLC",1.0,Sand Point,AK,Aleutians East,2,Petroleum Liquids,IC,NaN,...,NaN,,,,NaN,,,N,NaN,NaN
2,63560,"Sand Point Generating, LLC",1.0,Sand Point,AK,Aleutians East,3,Petroleum Liquids,IC,NaN,...,NaN,,,,NaN,,,N,NaN,NaN
3,63560,"Sand Point Generating, LLC",1.0,Sand Point,AK,Aleutians East,5.1,Petroleum Liquids,IC,NaN,...,NaN,,,,NaN,,,N,N,N
4,63560,"Sand Point Generating, LLC",1.0,Sand Point,AK,Aleutians East,WT1,Onshore Wind Turbine,WT,NaN,...,NaN,,,,NaN,,,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27914,67082,"Headwater Energy, LLC",70432.0,"Bishopville Solar, LLC",SC,Lee,27,Solar Photovoltaic,PV,NaN,...,NaN,,,,NaN,,,NaN,NaN,NaN
27915,67082,"Headwater Energy, LLC",70433.0,"Ebbie Solar, LLC",SC,Marion,28,Solar Photovoltaic,PV,NaN,...,NaN,,,,NaN,,,NaN,NaN,NaN
27916,67082,"Headwater Energy, LLC",70434.0,"OWC Energy Holdings, LLC",CA,Fresno,29,Solar Photovoltaic,PV,NaN,...,NaN,,,,NaN,,,NaN,NaN,NaN
27917,68005,Peppertown Road Solar LLC,70438.0,VA - Ameresco - Mechanicsville,VA,Hanover,MECH1,Solar Photovoltaic,PV,NaN,...,NaN,,,,NaN,,,NaN,NaN,NaN


In [21]:
#concatenate current and proposed generators to new dataframe
generators_df = pd.concat([current_gen_df, proposed_gen_df], axis=0)